In [ ]:
CATALOG = "spotify_etl"
SCHEMA = "bronze"
TABLE = ""  # set per notebook

dbutils.widgets.text("raw_base_path", "/Workspace/Users/pacioianu4@gmail.com/Files/spotify-end-to-end-api-project/data/raw", "RAW base path")
RAW_BASE_PATH = dbutils.widgets.get("raw_base_path").rstrip("/")

In [ ]:
import os, json

def collect():
    rows = []
    entity_path = f"{RAW_BASE_PATH}/artists_bulk"
    if not os.path.exists(entity_path): return rows
    for root, dirs, files in os.walk(entity_path):
        for fn in files:
            if fn.startswith("page_") and fn.endswith(".json") and not fn.endswith("_meta.json"):
                with open(os.path.join(root, fn), "r", encoding="utf-8") as f:
                    data = json.load(f)
                for artist in data.get("artists", []):
                    if not artist: continue
                    fo = artist.get("followers")
                    rows.append({
                        "artist_id": artist.get("id"), "artist_name": artist.get("name"),
                        "genres": artist.get("genres"),
                        "followers": fo.get("total") if isinstance(fo, dict) else None,
                        "popularity": artist.get("popularity"), "uri": artist.get("uri"),
                    })
    return rows

rows = collect()
if rows:
    df = spark.createDataFrame(rows).dropDuplicates(["artist_id"])
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
    print(f"Wrote {df.count()} rows to {CATALOG}.{SCHEMA}.{TABLE}")
else:
    print(f"No data for {TABLE}")
